## Ejercicio 7: Debugging — Errores encontrados

1. **Error 1 — Falta configuración de `SparkSession` y falta obtener `SparkContext` (`sc`):**
   Se crea `spark` sin nombre de aplicación (`.appName()`) ni configuración, y en la siguiente línea se usa `sc.textFile(...)` sin haber definido `sc = spark.sparkContext`, lo que provoca un `NameError: name 'sc' is not defined`.

2. **Error 2 — Ruta incorrecta para archivo local (`"./datos.csv"`):**
   En un entorno con Hadoop/HDFS configurado, una ruta relativa sin esquema hace que Spark intente buscar el archivo en HDFS en lugar del disco local. Para archivos locales se debe usar el prefijo `file://` acompañado de la ruta absoluta (`os.path.abspath(...)`).

3. **Error 3 — Uso peligroso de `collect()` en un dataset grande:**
   `rdd.collect()` trae absolutamente todos los registros de los *executors* a la memoria RAM del nodo *Driver*. Con un archivo grande provoca un `OutOfMemoryError` (OOM). Debe sustituirse por `take(n)` para inspeccionar una muestra segura o por operaciones distribuidas como `foreach()` / `saveAsTextFile()`.

4. **Error 4 — No se cierra la sesión de Spark:**
   Falta llamar a `spark.stop()` al finalizar. No cerrar la sesión deja procesos JVM y memoria RAM reservados innecesariamente en el clúster o contenedor.

In [1]:
import os, cloudpickle, pyspark.serializers
pyspark.serializers.cloudpickle = cloudpickle
from pyspark.sql import SparkSession

# Corrección 1: Configurar SparkSession y obtener SparkContext
spark = SparkSession.builder \
    .appName("Ejercicio7_DebuggingCorregido") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()

sc = spark.sparkContext

# Corrección 2: Construir ruta absoluta local con prefijo file://
ruta_absoluta = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd = sc.textFile(ruta_absoluta)

# Corrección 3: Usar take() en lugar de collect() para evitar saturar la memoria del Driver
muestra = rdd.take(5)
print("Muestra segura de las primeras 5 líneas:")
for linea in muestra:
    print(linea)

# Corrección 4: Cerrar la sesión de Spark correctamente
spark.stop()
print("\nSesión de Spark cerrada correctamente.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/20 02:52:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Muestra segura de las primeras 5 líneas:
name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000

Sesión de Spark cerrada correctamente.
